# z712 — Ensamble multi-semilla del residuo de Natalia

**Qué corre.** Tres combinaciones de su notebook, con cinco semillas cada una:

| combo | baseline | esquema |
|---|---|---|
| 1 | `ma_pond` | `D_lgbm_residuo` |
| 2 | `ma3` | `D_lgbm_residuo` |
| 3 | `tn0` | `B_lgbm_nivel` |


**Qué NO se toca.** La lógica de Natalia se ejecuta **verbatim**. Este notebook no reimplementa
nada: parchea su celda de Palancas, ejecuta su notebook tal cual con `nbclient`, y recolecta el
CSV. Si su pipeline cambia, esto sigue funcionando.

**Qué agrega, aprovechando que hay cómputo de sobra:**

1. **Cinco semillas por combo.** Es la única mejora sin contraindicación que quedó de todo lo
   aprendido: promediar baja la varianza sin tocar el sesgo. Medido en el track de AutoGluon, el
   ruido entre semillas de una misma configuración era 4,8 % de L1 relativo.
2. **Ejecución en paralelo.** Cada corrida es un proceso aparte con `OMP_NUM_THREADS` acotado,
   así N corridas usan la máquina entera sin pelearse. 15 corridas secuenciales son ~3,5 h; con
   4 workers, menos de una.
3. **Más Optuna** (60 trials en vez de 40), que es en lo que conviene gastar el cómputo extra.
4. **Diagnóstico del pool**: divergencia entre semillas del mismo combo, entre combos, y curva
   de convergencia. Si sumar semillas ya no mueve la aguja, se ve.

**Sin multiplicador.** Siete puntos de leaderboard en dos pipelines distintos, cero excepciones:
aplicar un multiplicador siempre perdió. Los CSV van crudos.

**Reanudable.** Una corrida = un CSV. Si ya existe, se saltea.

In [ ]:
%pip install -q nbclient nbformat ipykernel polars pandas numpy lightgbm optuna scikit-learn pyarrow

In [ ]:
# ruff: noqa: E402
import itertools
import json
import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/.drive")
    os.makedirs("/content/buckets", exist_ok=True)
    if not os.path.islink("/content/buckets/b1"):
        os.symlink("/content/.drive/My Drive/labo3", "/content/buckets/b1")
    os.environ["LABO3_BUCKET"] = "/content/buckets/b1"


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env); p.mkdir(parents=True, exist_ok=True); return p
    for c in ("/home/jupyter/buckets/b1", "/content/buckets/b1", "~/buckets/b1"):
        p = Path(c).expanduser()
        if p.exists():
            return p
    p = Path.cwd() / "bucket"; p.mkdir(parents=True, exist_ok=True); return p


BUCKET = resolver_bucket()
N_CORES = os.cpu_count() or 8
os.environ["LABO3_BUCKET"] = str(BUCKET)
print(f"bucket: {BUCKET} | cores: {N_CORES}")

## 1 — Palancas

In [ ]:
PARAM = {
    "notebook_base": "07_Residuo_sobre_baseline.ipynb",   # el de Natalia, se ejecuta verbatim
    "combos": [
        ("ma_pond", "D_lgbm_residuo"),
        ("ma3", "D_lgbm_residuo"),
        ("tn0", "B_lgbm_nivel"),
    ],
    "semillas": [102191, 314159, 777773, 116269, 241511],

    "n_trials": 60,          # el original usaba 40; el computo extra rinde mas aca que en semillas
    "techo_arboles": 800,
    "workers": 4,            # corridas en paralelo (se ajusta solo si no alcanzan los nucleos)
    "hilos_min": 4,          # hilos por corrida. Con 1 hilo LightGBM va ~8x mas lento
    "timeout_min": 90,       # por corrida

    "kaggle_competition": "labo-iii-2026-ba",
    "submit": True,
    "sufijo": "",
}

DIR_EXP = BUCKET / "exp" / ("z712_natalia_multisemilla" + PARAM["sufijo"])
DIR_RUNS = DIR_EXP / "corridas"
DIR_NB = DIR_EXP / "notebooks"
DIR_OUT = DIR_EXP / "submits"
for d in (DIR_RUNS, DIR_NB, DIR_OUT):
    d.mkdir(parents=True, exist_ok=True)

# el notebook de Natalia: al lado de este, o en el bucket
CANDIDATOS_NB = [Path.cwd() / PARAM["notebook_base"],
                 BUCKET / PARAM["notebook_base"],
                 BUCKET / "notebooks" / PARAM["notebook_base"]]
NB_BASE = next((p for p in CANDIDATOS_NB if p.exists()), None)
if NB_BASE is None:
    raise SystemExit(
        f"\n>>> No encuentro '{PARAM['notebook_base']}'.\n"
        f">>> Buscado en: {[str(p) for p in CANDIDATOS_NB]}\n"
        f">>> Copialo al lado de este notebook o al bucket.\n")

# Su notebook NO descarga los datos: los espera en el bucket. Se chequea antes de lanzar 15
# corridas que fallarian todas por lo mismo.
FALTAN = [a for a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt")
          if not (BUCKET / "datasets" / a).exists()]
if FALTAN:
    print(f"faltan datasets {FALTAN}, los bajo a {BUCKET / 'datasets'}")
    (BUCKET / "datasets").mkdir(parents=True, exist_ok=True)
    for a in FALTAN:
        subprocess.run(["wget", "-q",
                        f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{a}",
                        "-O", str(BUCKET / "datasets" / a)], check=True)
    print("descargados")

# El kernel que va a usar nbclient tiene que existir
_k = subprocess.run([sys.executable, "-m", "jupyter", "kernelspec", "list"],
                    capture_output=True, text=True)
if "python3" not in _k.stdout:
    print(">>> No hay kernel 'python3' registrado. Corriendo:")
    print(">>>   python -m ipykernel install --user --name python3")
    subprocess.run([sys.executable, "-m", "ipykernel", "install", "--user", "--name", "python3"],
                   check=False)

PLAN = [{"tag": f"{b}__{e}__s{s}", "baseline": b, "esquema": e, "semilla": s,
         "combo": f"{b}__{e}"}
        for (b, e), s in itertools.product(PARAM["combos"], PARAM["semillas"])]
print(f"notebook base: {NB_BASE}")
print(f"plan: {len(PLAN)} corridas ({len(PARAM['combos'])} combos x {len(PARAM['semillas'])} semillas)")
for c in PARAM["combos"]:
    print(f"   {c[0]:8s} + {c[1]}")

## 2 — Parcheo de las Palancas

Se toca **sólo** la celda `PARAM` de Natalia: baseline, esquema, semilla, trials, competencia y
`submit=False` (los submits los maneja este notebook, no el suyo). El `sufijo` le da a cada
corrida su propia carpeta, para que no se pisen entre sí.

Si alguno de los reemplazos no encuentra su texto, **corta**: es preferible fallar a ejecutar
15 corridas con la configuración equivocada, que es exactamente lo que pasa cuando un parche
silencioso no aplica.

In [ ]:
REEMPLAZOS = [
    ("'baseline': 'auto',", "'baseline': '{baseline}',"),
    ("'esquema': 'auto',", "'esquema': '{esquema}',"),
    ("'n_trials': 40,", "'n_trials': {n_trials},"),
    ("'kaggle_competition': 'labo-iii-2026-rosario',", "'kaggle_competition': 'labo-iii-2026-ba',"),
    ("'submit': False,", "'submit': False,"),
    ("'semillas_ensemble': [102191],", "'semillas_ensemble': [{semilla}],"),
    ("'semilla': 102191,", "'semilla': {semilla},"),
    ("'sufijo': '',", "'sufijo': '_s{semilla}',"),
]


def preparar(cfg: dict) -> Path:
    nb = nbformat.read(NB_BASE, as_version=4)
    celda = next((c for c in nb.cells if c.cell_type == "code" and "PARAM = {" in c.source), None)
    if celda is None:
        raise RuntimeError("no encontre la celda PARAM en el notebook de Natalia")
    src, faltan = celda.source, []
    for viejo, nuevo in REEMPLAZOS:
        if viejo not in src:
            faltan.append(viejo)
            continue
        src = src.replace(viejo, nuevo.format(**cfg, n_trials=PARAM["n_trials"]), 1)
    if faltan:
        raise RuntimeError(
            f"el notebook de Natalia cambio: no encontre {faltan}. "
            f"Actualiza REEMPLAZOS antes de correr, o vas a ejecutar 15 veces la config equivocada.")
    celda.source = src
    destino = DIR_NB / f"{cfg['tag']}.ipynb"
    nbformat.write(nb, destino)
    return destino


_p = preparar(PLAN[0])
print("parcheo verificado sobre la primera corrida ->", _p.name)
import nbformat as _nbf
_cel = next(c for c in _nbf.read(_p, as_version=4).cells if c.cell_type == "code" and "PARAM = {" in c.source)
for _l in _cel.source.split("\n"):
    if any(k in _l for k in ["'baseline'", "'esquema'", "'semilla'", "'sufijo'", "'n_trials'",
                             "kaggle_competition", "'submit'"]):
        print("   ", _l.strip())

## 3 — Ejecución en paralelo

Cada corrida va en su propio proceso, con `OMP_NUM_THREADS` acotado para que los workers no se
peleen por los núcleos. El log completo queda en `notebooks/<tag>.log`; si una falla, se reporta
y las demás siguen.

In [ ]:
CORRE = r'''
import sys
from nbclient import NotebookClient
import nbformat
nb = nbformat.read(sys.argv[1], as_version=4)
NotebookClient(nb, timeout=int(sys.argv[2]), kernel_name="python3",
               resources={"metadata": {"path": sys.argv[3]}}).execute()
nbformat.write(nb, sys.argv[1])
print("EJECUTADO OK")
'''
RUTA_CORRE = DIR_EXP / "_ejecutar.py"
RUTA_CORRE.write_text(CORRE)

# Cada worker necesita varios hilos: LightGBM con 1 hilo tarda casi un orden de magnitud mas, y
# la paralelizacion termina costando mas de lo que da. Se respeta un piso de `hilos_min`.
WORKERS = max(1, min(PARAM["workers"], N_CORES // PARAM["hilos_min"]))
HILOS = max(PARAM["hilos_min"], N_CORES // max(1, WORKERS))
if WORKERS != PARAM["workers"]:
    print(f"AVISO: bajo workers de {PARAM['workers']} a {WORKERS} para que cada uno tenga "
          f"{HILOS} hilos ({N_CORES} nucleos disponibles)")


def csv_de(cfg: dict) -> Path:
    return DIR_RUNS / f"{cfg['tag']}.csv"


def una_corrida(cfg: dict) -> dict:
    fsal = csv_de(cfg)
    if fsal.exists():
        return {**cfg, "estado": "ya estaba", "seg": 0}
    t0 = time.time()
    nb_path = preparar(cfg)
    log = DIR_NB / f"{cfg['tag']}.log"
    env = {**os.environ, "OMP_NUM_THREADS": str(HILOS), "MKL_NUM_THREADS": str(HILOS),
           "LABO3_BUCKET": str(BUCKET)}
    with open(log, "w") as fl:
        r = subprocess.run([sys.executable, str(RUTA_CORRE), str(nb_path),
                            str(PARAM["timeout_min"] * 60), str(DIR_NB)],
                           stdout=fl, stderr=subprocess.STDOUT, env=env)
    if r.returncode != 0:
        return {**cfg, "estado": f"FALLO (ver {log.name})", "seg": time.time() - t0}

    # el notebook de Natalia deja submission_<objetivo>.csv en exp_residuo/<EXPERIMENTO>/
    cands = sorted((BUCKET / "exp_residuo").glob(f"*_s{cfg['semilla']}*/submission_*.csv"))
    cands = [c for c in cands if cfg["baseline"] in c.parent.name and cfg["esquema"] in c.parent.name]
    if not cands:
        return {**cfg, "estado": "sin CSV de salida", "seg": time.time() - t0}
    shutil.copyfile(cands[-1], fsal)
    return {**cfg, "estado": "ok", "seg": time.time() - t0}


t_ini = time.time()
pend = [c for c in PLAN if not csv_de(c).exists()]
print(f"{len(PLAN) - len(pend)}/{len(PLAN)} ya estaban. Lanzando {len(pend)} corridas "
      f"con {WORKERS} workers x {HILOS} hilos.")
print(f"log de cada una en {DIR_NB}/<tag>.log  --  segui el avance con:")
print(f"  tail -f {DIR_NB}/*.log\n")

resultados = [{**c, "estado": "ya estaba", "seg": 0} for c in PLAN if csv_de(c).exists()]
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futuros = {ex.submit(una_corrida, c): c for c in pend}
    for i, fut in enumerate(as_completed(futuros), 1):
        r = fut.result()
        resultados.append(r)
        print(f"  [{i:2d}/{len(pend)}] {r['tag']:34s} {r['estado']:22s} "
              f"{r['seg'] / 60:5.1f} min  (transcurrido {(time.time() - t_ini) / 60:.0f} min)")
fallos = [r for r in resultados if r["estado"] not in ("ok", "ya estaba")]
print(f"\ntotal {(time.time() - t_ini) / 60:.0f} min | ok: {len(resultados) - len(fallos)}/{len(resultados)}")
if fallos:
    print("FALLARON:", [r["tag"] for r in fallos])

## 4 — Diagnóstico del pool

Dos preguntas. **¿Las semillas de un mismo combo difieren?** Si no, promediarlas no aporta y
sobran cuatro corridas. **¿Los combos difieren entre sí más que las semillas?** Si sí, mezclar
combos aporta más que sumar semillas, y ahí conviene gastar el cómputo.

In [ ]:
POOL = {}
for cfg in PLAN:
    f = csv_de(cfg)
    if f.exists():
        POOL[cfg["tag"]] = (pd.read_csv(f).set_index("product_id").iloc[:, 0].sort_index(),
                            cfg["combo"])
if not POOL:
    raise RuntimeError("no hay ninguna corrida en el pool")
IDX = next(iter(POOL.values()))[0].index
for k, (v, _) in POOL.items():
    assert (v.index == IDX).all(), f"{k} tiene otro conjunto de product_id"

TAGS = list(POOL)
M = np.array([POOL[t][0].values for t in TAGS])
COMBO = np.array([POOL[t][1] for t in TAGS])
print(f"pool: {len(TAGS)} corridas x {len(IDX)} productos\n")
print(pd.DataFrame({"tag": TAGS, "total_tn": M.sum(1).round(0)}).to_string(index=False))


def div(a, b):
    return float(np.abs(a - b).sum() / ((a + b).sum() / 2))


print("\ndivergencia L1 relativa:")
intra = [div(M[i], M[j]) for i, j in itertools.combinations(range(len(TAGS)), 2)
         if COMBO[i] == COMBO[j]]
inter = [div(M[i], M[j]) for i, j in itertools.combinations(range(len(TAGS)), 2)
         if COMBO[i] != COMBO[j]]
if intra:
    print(f"  entre semillas del MISMO combo : {np.mean(intra):.4f}")
if inter:
    print(f"  entre combos DISTINTOS         : {np.mean(inter):.4f}")
if intra and inter:
    print("  -> " + ("mezclar combos aporta mas que sumar semillas"
                     if np.mean(inter) > 1.5 * np.mean(intra)
                     else "las dos fuentes de diversidad son parecidas"))

cons = M.mean(0)
print("\nconvergencia: distancia media al consenso usando k corridas al azar")
rng = np.random.default_rng(102191)
for k in range(1, len(TAGS) + 1):
    combos = list(itertools.combinations(range(len(TAGS)), k))
    if len(combos) > 200:
        combos = [tuple(rng.choice(len(TAGS), k, replace=False)) for _ in range(200)]
    print(f"  k={k:2d}  {np.mean([div(M[list(c)].mean(0), cons) for c in combos]):.4f}")

## 5 — Ensambles

**Media, no mediana.** Con cinco miembros por combo la mediana descarta tres quintos de la
información por producto y sólo paga si una corrida explota; con pools chicos la media usa todo.

Tres candidatos: cada combo promediado por separado (para poder compararlos en el leaderboard) y
dos mezclas. El total en toneladas se contrasta contra la **banda de febrero, 28 400 – 30 600**,
calculada sobre los febreros reales de los 780 corrigiendo por los 120 productos que no existían
en feb-2019.

In [ ]:
BANDA = (28400, 30600)


def escribir(nombre: str, cols: list, desc: str):
    v = np.maximum(np.array([POOL[c][0].values for c in cols]).mean(0), 0.0)
    f = DIR_OUT / f"z712_{nombre}.csv"
    pd.DataFrame({"product_id": IDX.values, "tn": v}).to_csv(f, index=False)
    est = "dentro" if BANDA[0] <= v.sum() <= BANDA[1] else ("ALTO" if v.sum() > BANDA[1] else "BAJO")
    print(f"  {nombre:26s} n={len(cols):2d}  {v.sum():9.0f} tn  [{est}]  {desc}")
    return f, v


print("ensambles:")
SUBMITS = []
por_combo = {}
for b, e in PARAM["combos"]:
    cols = [t for t in TAGS if POOL[t][1] == f"{b}__{e}"]
    if cols:
        f, v = escribir(f"{b}_{e}", cols, "un combo, promedio de semillas")
        por_combo[f"{b}__{e}"] = v
        SUBMITS.append((f, f"z712 {b}+{e}, media de {len(cols)} semillas"))

if len(por_combo) > 1:
    f, _ = escribir("mezcla_combos", TAGS, "todas las corridas, peso igual")
    SUBMITS.append((f, f"z712 mezcla de {len(TAGS)} corridas ({len(por_combo)} combos)"))
    v = np.maximum(np.mean(list(por_combo.values()), axis=0), 0.0)
    f2 = DIR_OUT / "z712_mezcla_por_combo.csv"
    pd.DataFrame({"product_id": IDX.values, "tn": v}).to_csv(f2, index=False)
    print(f"  {'mezcla_por_combo':26s} n={len(por_combo):2d}  {v.sum():9.0f} tn"
          f"  [{'dentro' if BANDA[0] <= v.sum() <= BANDA[1] else 'fuera'}]"
          f"  cada combo pesa igual, no cada corrida")
    SUBMITS.append((f2, f"z712 mezcla equiponderada de {len(por_combo)} combos"))

(DIR_EXP / "resumen.json").write_text(json.dumps({
    "plan": len(PLAN), "ok": len(TAGS), "combos": [list(c) for c in PARAM["combos"]],
    "semillas": PARAM["semillas"], "n_trials": PARAM["n_trials"],
    "div_intra_combo": float(np.mean(intra)) if intra else None,
    "div_inter_combo": float(np.mean(inter)) if inter else None,
    "totales": {t: float(POOL[t][0].sum()) for t in TAGS}}, indent=2))

In [ ]:
def kaggle_submit(archivo: Path, msg: str):
    flag = archivo.with_suffix(".done")
    if flag.exists():
        print("ya subido:", archivo.name); return
    r = subprocess.run(["kaggle", "competitions", "submit", "-c", PARAM["kaggle_competition"],
                        "-f", str(archivo), "-m", msg], capture_output=True, text=True)
    print(archivo.name, "->", (r.stdout or r.stderr).strip()[:160])
    if r.returncode == 0:
        flag.write_text(msg)


if PARAM["submit"]:
    for f, msg in SUBMITS:
        kaggle_submit(f, msg)
else:
    print("submit=False -> CSVs en", DIR_OUT)